## Análisis exploratorio de la caida en la Competencia Lectora de los informes PISA en los últimos años  (2018 -> 2022 -> 2025)

## 1. Introducción

PISA es la evaluación internacional de la OCDE que mide cada tres años las competencias
del alumnado de 15 años. En la Comunidad Valenciana, la puntuación media en lectura pasó
de 481,8 en 2022 a 423,5 en 2025. Este proyecto explora **si esa caída puede explicarse
por cambios en el perfil del alumnado o en el uso de tecnología en el centro**.

**Pregunta:** 
¿Qué factores observables en los microdatos de PISA acompañan a la caída
de la competencia lectora 2022-2025 en la Comunidad Valenciana?
Dentro de la Comunidad Valenciana, para el año de 2025, ¿que variables están más relacionadas con una menor nota en lectura?

**Hipótesis de partida**:
- H1. El nivel socioeconómico (ESCS) del alumnado condiciona la nota en la competencia lectora.
- H2. La titularidad del centro (público/privado) del alumnado condiciona la nota en la competencia lectora.
- H3. Han cambiado la composición por origen migratorio y el idioma de la prueba.

**Alcance y límites:** análisis exploratorio, descriptivo y sin conclusiones causales,
con una muestra de unos 50 centros por ciclo.

**Hipótesis que no da tiempo a estudiar**
- H5. El tiempo de pantalla invertido en los centros ha aumentado, asociandose con la caida en la competencia lectora.
- H6. El tiempo de pantalla invertido por los alumnos fuera del aula ha aumentado, asociandose con la caida en la competencia lectora.
- H7. Las metodologías docentes en los centros ha cambiado, influyendo en la caida en la competencia lectora.

## 2. Fuentes de datos

Microdatos del cuestionario de alumnado de PISA 2018, 2022 y 2025 (OCDE, ficheros SPSS):
`CY07_MSU_STU_QQQ.sav`, `CY08MSP_STU_QQQ.SAV` y `CY09_MS_STU_PUF.sav`, y los codebooks correcpondientes. [https://www.oecd.org/en/data/datasets/pisa-2018-database.html#codebook]

- **Unidad de análisis:** alumno de 15 años. 
- **Diseño muestral:** primero centros y luego alumnos dentro de cada centro, por lo que hay que usar pesos (`W_FSTUWT`) y pesos replicados (`W_FSTURWT1-80`) para los errores.
- **Competencia lectora:** 10 valores plausibles (`PV1READ`–`PV10READ`).
- **Muestra de la Comunidad Valenciana:** 1.753 / 1.625 / 1.712 alumnos en 53 / 53 / 54
  centros (2018 / 2022 / 2025).
- **Variables de contexto:** ESCS, origen migratorio (`IMMIG`), idioma hablado en casa (`LANGN`),
  idioma en que se realiza la prueba (`LANGTEST_COG`), horas de uso digital en el centro (`ST326Q01JA`).

In [10]:
#Librerias
import pyreadstat
import pandas as pd
import numpy as np

In [ ]:
# Definimos las variables que estudiaremos en el análisis
cols = (["CNT","STRATUM", "CNTSCHID", "CNTSTUID",
            "ST326Q01JA", "ESCS", "W_FSTUWT", "LANGN", "LANGTEST_COG", "IMMIG"]
           + [f"PV{i}READ" for i in range(1, 11)] + [f"W_FSTURWT{i}" for i in range(1, 81)])

cols_pantallas_18 = [f"ST176Q0{i}IA" for i in range(1, 8) ]

cols18 = cols+cols_pantallas_18

In [4]:
# Cargamos los datos filtrados por las columnas que queremos y por la comunidad valenciana

archivo_pisa25 = 'data/CY09_MS_STU_PUF.sav'
print("Cargando los archivo de microdatos de PISA (puede tardar unos minutos)...")
codigos_valencia25 = ['ESP33', 'ESP34']

archivo_pisa22="data/CY08MSP_STU_QQQ.SAV"
print("Cargando los archivo de microdatos de PISA 2022 (puede tardar unos minutos)...")
codigos_valencia22 = ['ESP34', 'ESP35']

archivo_pisa18="data/CY07_MSU_STU_QQQ.sav"
print("Cargando los archivo de microdatos de PISA 2018 (puede tardar unos minutos)...")
codigos_valencia18 = ['ESP9038', 'ESP1738', 'ESP9039', 'ESP1739']

def cargar_valencia(ruta, estratos, cols):
    # 1) Leemos solo las columnas que necesitamos
    df, meta = pyreadstat.read_sav(ruta, usecols=cols)

    # 2) Nos quedamos con los alumnos de España y de los estratos de Valencia
    d = df[(df["CNT"] == "ESP") & (df["STRATUM"].isin(estratos))]
    
    # 3) Algunas comprobaciones en la práctica: tamaño y nº de pesos replicados
    n_rep = len([c for c in d.columns if "RWT" in c])
    print("Filas:", d.shape[0], "| centros:", d["CNTSCHID"].nunique(), "| réplicas:", n_rep)
    return d, meta

df_valencia25, meta25 = cargar_valencia(archivo_pisa25, codigos_valencia25, cols)
df_valencia22, meta22 = cargar_valencia(archivo_pisa22, codigos_valencia22, cols)
df_valencia18, meta18 = cargar_valencia(archivo_pisa18, codigos_valencia18, cols18)



Cargando los archivo de microdatos de PISA (puede tardar unos minutos)...
Cargando los archivo de microdatos de PISA 2022 (puede tardar unos minutos)...
Cargando los archivo de microdatos de PISA 2018 (puede tardar unos minutos)...
Filas: 1712 | centros: 54 | réplicas: 80
Filas: 1625 | centros: 53 | réplicas: 80
Filas: 1753 | centros: 53 | réplicas: 80


In [5]:
# Pasamos a csv los df cargados y creados 
df_valencia25.to_csv("data/pisa_valencia25.csv")
df_valencia22.to_csv("data/pisa_valencia22.csv")
df_valencia18.to_csv("data/pisa_valencia18.csv")


A. Verificamos que los estratos escogidos son los de Valencia :

In [6]:
for anio, m, cods in [("2018", meta18, codigos_valencia18),
                      ("2022", meta22, codigos_valencia22),
                      ("2025", meta25, codigos_valencia25)]:
    etiquetas = m.variable_value_labels.get("STRATUM", {})
    for c in cods:
        print(anio, c, "->", etiquetas.get(c, "SIN ETIQUETA"))

2018 ESP9038 -> ESP - stratum 38: Valencia, public
2018 ESP1738 -> ESP - stratum 38: Valencia, public
2018 ESP9039 -> ESP - stratum 39: Valencia, private
2018 ESP1739 -> ESP - stratum 39: Valencia, private
2022 ESP34 -> ESP - stratum 34: Comunidad Valenciana,Public
2022 ESP35 -> ESP - stratum 35: Comunidad Valenciana,Private
2025 ESP33 -> Comunidad Valenciana, Public
2025 ESP34 -> Comunidad Valenciana, Private


In [8]:
# Comprobaciones iniciales

dfs = {"2018": df_valencia18, "2022": df_valencia22, "2025": df_valencia25}
resumen = pd.DataFrame({
    a: {"alumnos": len(d),
        "centros": d["CNTSCHID"].nunique(),  # cuantos valores distintos hay 
        "alumnos duplicados": d["CNTSTUID"].duplicated().sum(),
        "pesos nulos": d["W_FSTUWT"].isna().sum()}
    for a, d in dfs.items()})
print(resumen)

                    2018  2022  2025
alumnos             1753  1625  1712
centros               53    53    54
alumnos duplicados     0     0     0
pesos nulos            0     0     0


## Resumen de las decisiones de limpieza, que se llevará a cabo en cada uno de los análisis de cada variable:
- No se imputan valores perdidos: se excluyen solo en el cálculo que usa cada variable.
- No se eliminan centros pequeños ni alumnos atípicos: los pesos ya recogen el diseño.
- No se eliminan duplicados porque no hay ninguno (CNTSTUID único).

A continuación dedicamos un espacio a búsquedas concretas en las etiquetas

In [ ]:
for a, df, m in [("2018", df_valencia18, meta18),
                 ("2022", df_valencia22, meta22),
                 ("2025", df_valencia25, meta25)]:
    print("-----", a)
    esta = "ST326Q01JA" in df.columns
    print("¿Está la columna?", esta)
    if esta:
        print("Etiqueta:", m.column_names_to_labels.get("ST326Q01JA"))
        print("Categorías:", m.variable_value_labels.get("ST326Q01JA"))
        print("Con respuesta:", df["ST326Q01JA"].notna().sum(), "de", len(df))

----- 2018
¿Está la columna? False
----- 2022
¿Está la columna? True
Etiqueta: This school year, how many hours/day use [digital resources] for: For learning activities at school
Categorías: {1.0: 'None', 2.0: 'Up to 1 hour', 3.0: 'More than 1 hour and up to 2 hours', 4.0: 'More than 2 hours and up to 3 hours', 5.0: 'More than 3 hours and up to 4 hours', 6.0: 'More than 4 hours and up to 5 hours', 7.0: 'More than 5 hours and up to 6 hours', 8.0: 'More than 6 hours and up to 7 hours', 9.0: 'More than 7 hours', 95.0: 'Valid Skip', 97.0: 'Not Applicable', 98.0: 'Invalid', 99.0: 'No Response'}
Con respuesta: 1579 de 1625
----- 2025
¿Está la columna? True
Etiqueta: This school year, about how many hours a day do you usually use <digital resources> in the following situations? - For learning activities at school
Categorías: {1.0: 'None', 2.0: 'Up to 1 hour', 3.0: 'More than 1 hour and up to 2 hours', 4.0: 'More than 2 hour and up to 3 hours', 5.0: 'More than 3 hour and up to 4 hours', 6.0: '

In [44]:
for a, df in [("2022", df_valencia22), ("2025", df_valencia25)]:
    print(a, df["ST326Q01JA"].value_counts(dropna=False).sort_index().to_dict())
    m = df.groupby("CNTSCHID")["ST326Q01JA"].apply(lambda s: s.isna().mean())
    print("   centros con más del 40% sin respuesta:", (m > 0.4).sum(), "de", len(m))

2022 {1.0: 325, 2.0: 627, 3.0: 288, 4.0: 140, 5.0: 50, 6.0: 44, 7.0: 41, 8.0: 32, 9.0: 32, nan: 46}
   centros con más del 40% sin respuesta: 0 de 53
2025 {1.0: 314, 2.0: 558, 3.0: 282, 4.0: 112, 5.0: 64, 6.0: 38, 7.0: 36, 8.0: 35, 9.0: 34, nan: 239}
   centros con más del 40% sin respuesta: 3 de 54


In [45]:
def dist_uso(df):
    d = df[df["ST326Q01JA"].between(1, 9)].dropna(subset=["W_FSTUWT"])
    tramo = pd.cut(d["ST326Q01JA"], bins=[0, 1, 2, 3, 9],
                   labels=["Ninguno", "Hasta 1 h", "1-2 h", "Más de 2 h"])
    w = d["W_FSTUWT"]
    return w.groupby(tramo, observed=True).sum() / w.sum() * 100

print(pd.DataFrame({"2022": dist_uso(df_valencia22), "2025": dist_uso(df_valencia25)}).round(1))

            2022  2025
ST326Q01JA            
Ninguno     20.8  21.6
Hasta 1 h   40.2  37.7
1-2 h       18.2  18.9
Más de 2 h  20.8  21.8


In [18]:
# 1) Solo los que hablan español en casa (código 156)
esp_casa = df_valencia22[df_valencia22["LANGN"] == 156]

# 2) Qué proporción hizo la prueba en cada idioma
print(esp_casa["LANGTEST_COG"].value_counts())
print((esp_casa["LANGTEST_COG"].value_counts(normalize=True) * 100).round(1))

# 3) Tabla completa, por si quieres ver todos los grupos a la vez
print(pd.crosstab(df_valencia22["LANGN"], df_valencia22["LANGTEST_COG"], margins=True))

LANGTEST_COG
156.0    1152
608.0      10
160.0       1
Name: count, dtype: int64
LANGTEST_COG
156.0    99.1
608.0     0.9
160.0     0.1
Name: proportion, dtype: float64
LANGTEST_COG  156.0  160.0  608.0   All
LANGN                                  
156.0          1152      1     10  1163
160.0             8      1      1    10
451.0             1      0      0     1
474.0             1      0      0     1
608.0           220      6     57   283
668.0             1      0      0     1
852.0           140      0      2   142
999.0            24      0      0    24
All            1547      8     70  1625


In [52]:
etq_test = meta25.variable_value_labels.get("LANGN", {})
etq_casa = meta25.variable_value_labels.get("LANGN", {})

print({k: v for k, v in etq_test.items() })
print({k: v for k, v in etq_casa.items() if k in (156.0, 160.0, 608.0, 852.0, 668.0,474.0,451.0 ,999.0, 690.0)})

{105.0: 'Kurdish (kur)', 108.0: 'Tagalog (tgl)', 112.0: 'Waray (Philippines) (war)', 113.0: 'Indonesian (ind)', 118.0: 'Romanian (rum)', 121.0: 'Estonian (est)', 130.0: 'Georgian (geo)', 137.0: 'Somali (som)', 140.0: 'Albanian (sqi)', 142.0: 'Albanian or the language of another Balkan country (akn)', 147.0: 'Amharic or Tigrinya (amt)', 148.0: 'German (deu)', 150.0: 'Afrikaans (afr)', 154.0: 'Tamil (tam)', 156.0: 'Spanish (spa)', 160.0: 'Catalan (cat)', 170.0: 'Slovak (slo)', 192.0: 'Bosnian (bos)', 195.0: 'Scots (sco)', 200.0: 'Italian (ita)', 202.0: 'Mongolian (mon)', 204.0: 'Malay (macrolanguage) (msa)', 232.0: 'Portuguese (por)', 237.0: 'Tajik (tgk)', 244.0: 'Czech (cze)', 246.0: 'Bikol (bik)', 254.0: 'Pangasinan (pag)', 258.0: 'Urdu (urd)', 264.0: 'Danish (dan)', 265.0: 'Faroese (fao)', 266.0: 'Croatian (hrv)', 272.0: 'Samoan (smo)', 273.0: 'Polish (pol)', 275.0: 'Kazakh (kaz)', 286.0: 'Japanese (jpn)', 301.0: 'Korean (kor)', 313.0: 'English (eng)', 316.0: 'Chinese (zho)', 317.0: '

In [ ]:
# Ver la etiqueta descriptiva que SPSS le da a cada código de estrato en 2018
etiquetas_estrato = meta25.variable_value_labels.get("STRATUM")

if etiquetas_estrato:
  for codigo in ["ESP33", "ESP34"]:  # O prueba los que quieras consultar
    print(f"Código {codigo}: {etiquetas_estrato.get(codigo, 'No encontrado')}")
else:
  print("No se encontraron etiquetas de valor integradas para STRATUM.")

Código ESP33: Comunidad Valenciana, Public
Código ESP34: Comunidad Valenciana, Private


In [17]:
etiquetas_estrato = meta25.variable_value_labels.get("STRATUM")

if etiquetas_estrato:
  print(
      "Buscando 'Valencia' o 'Valenciana' en los metadatos de STRATUM (2022)..."
  )
  encontrados = False
  for codigo, etiqueta in etiquetas_estrato.items():
    # Comprobamos si el texto de la etiqueta contiene valencia (sin importar mayúsculas/minúsculas)
    if any(
        palabra in str(etiqueta).lower() for palabra in ["valencia", "valenciana"]
    ):
      print(f" Código exacto: {codigo}  -->  Etiqueta: {etiqueta}")
      encontrados = True

  if not encontrados:
    print(
        "No se ha encontrado ninguna coincidencia con 'Valencia' en las"
        " etiquetas de STRATUM."
    )
else:
  print("No se encontraron etiquetas de valor integradas para STRATUM.")

Buscando 'Valencia' o 'Valenciana' en los metadatos de STRATUM (2022)...
 Código exacto: ESP33  -->  Etiqueta: Comunidad Valenciana, Public
 Código exacto: ESP34  -->  Etiqueta: Comunidad Valenciana, Private
